In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from helper import DataLoader
import models.SimpleGCN.SimpleGCN as SimpleGCN
import torch
from sklearn.metrics import root_mean_squared_error

In [4]:
DATAPATH = "../data"
RATING_DATA = DATAPATH + "/train_ratings.csv"
TBR_DATA = DATAPATH + "/train_tbr.csv"

In [5]:
data_manager = DataLoader.DataLoader(data_dir=DATAPATH)
train_df, valid_df = data_manager.load_and_split()

In [6]:
print(train_df)

         sid  pid  rating
0          0    1       5
1          0   13       4
2          0   22       5
3          0   26       4
4          0   62       4
...      ...  ...     ...
896738   957  337       3
896739  2218  232       5
896740  1146  255       4
896741  5758  519       4
896742  1062  437       5

[896743 rows x 3 columns]


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
edge_t0_index, edge_t0_weights, edge_t1_index, edge_t1_weights = data_manager.get_graph_tensors(device)

In [9]:
# model = DualGraphGCN.DualGraphGCN(
#     num_users=data_manager.num_users,
#     num_items=data_manager.num_items,
#     embedding_dim=64,
#     num_layers=2,
#     dropout=0.5
# ).to(device)

# # model = SimpleGCN.SimpleGCN(
# #     num_users=data_manager.num_users,
# #     num_items=data_manager.num_items,
# #     embedding_dim=64,
# #     num_layers=1,
# #     dropout=0.3
# # ).to(device)

# criterion = torch.nn.MSELoss()
# def predict_ratings(sids, pids):
#     model.eval()
#     with torch.no_grad():
#         s_tensor = torch.tensor(sids, dtype=torch.long).to(device)
#         p_tensor = torch.tensor(pids, dtype=torch.long).to(device)
#         # Pass all 4 graph tensors
#         out, _, _ = model(s_tensor, p_tensor, 
#                    edge_t0_index, edge_t0_weights, 
#                    edge_t1_index, edge_t1_weights)        
#         # out, _, _ = model(s_tensor, p_tensor, 
#         #             edge_t0_index, edge_t0_weights)
#         return out.cpu().numpy()

In [10]:
# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=1e-2,
# )

In [11]:
# EPOCHS = 5000
# LAMBDA_REG = 1e-4
# LAMBDA_WISHLIST = 0.5

# # --- 3. UPDATED TRAINING LOOP ---
# for epoch in range(EPOCHS):
#     model.train()
#     optimizer.zero_grad()
    
#     # Forward pass using both relation types
#     # We predict ratings for the users/items present in our t0 (ratings) edges
    
#     preds, (u_w, i_w), (u_init_r, i_init_r, u_init_w, i_init_w) = model(
#         edge_t0_index[0], edge_t0_index[1], 
#         edge_t0_index, edge_t0_weights,
#         edge_t1_index, edge_t1_weights
#     )    
#     pred_wishlist = (u_w[edge_t1_index[0]] * i_w[edge_t1_index[1]]).sum(-1)
#     # preds, u_init, i_init = model(
#     #     edge_t0_index[0],edge_t0_index[1], edge_t0_index, edge_t0_weights
#     # )
    
#     # Loss is still calculated against the actual ratings (weights_t0)
#     mse_loss = criterion(preds, edge_t0_weights * 5)
#     mse_loss_wishlist = criterion(pred_wishlist, edge_t1_weights * 1)
#     # l2_reg = (u_init.norm(2)**2 + i_init.norm(2)**2) / len(edge_t1_index[0])

#     loss = (
#         mse_loss 
#         + LAMBDA_WISHLIST * mse_loss_wishlist   # auxiliary
#         + LAMBDA_REG * sum(e.norm() for e in [u_init_r, i_init_r, u_init_w, i_init_w])  # regularization
#     )

    
#     loss.backward()
#     optimizer.step()
    
#     if epoch % 1 == 0:
#         val_preds = predict_ratings(valid_df["sid"].values, valid_df["pid"].values)
#         val_rmse = root_mean_squared_error(valid_df["rating"].values, val_preds)
#         print(f"Epoch {epoch} | Loss: {loss.item():.4f} | MSE: {mse_loss.item():.4f} | MSE Wishlist: {mse_loss_wishlist.item():.4f} | Val RMSE: {val_rmse:.4f}")


In [20]:
rating_model = SimpleGCN.SimpleGCN(data_manager.num_users, data_manager.num_items, embedding_dim=32, dropout=0.3, num_layers=2, layer_type="GCN")
rating_model = rating_model.to(device)
def rating_val():
    rating_model.eval()
    with torch.no_grad():
        # 1. Pass the IDs AND the graph tensors
        val_preds = rating_model.predict_ratings(
            valid_df["sid"].values, 
            valid_df["pid"].values,
            edge_t0_index,    # The graph structure
            edge_t0_weights   # The graph weights
        )
        
        # 2. Ensure variable names match (val_preds)
        return root_mean_squared_error(valid_df["rating"].values, val_preds * 5)




In [21]:
rating_model.fit(
    edge_index=edge_t0_index,
    weights=edge_t0_weights,
    log_every=50,
    lr=1e-2,
    val_fn=rating_val,
    targets=edge_t0_weights.float(),
    lambda_reg=1e-10,
    early_stop_patience=1000,
    scheduler_patience=25
)

Epoch     0 | Loss: 0.1130 | Task: 0.1130 | Monitor: 1.6767 | LR: 1.00e-02 | No improvement: 0/1000
Epoch    50 | Loss: 0.0368 | Task: 0.0368 | Monitor: 0.9425 | LR: 1.00e-02 | No improvement: 0/1000
Epoch   100 | Loss: 0.0361 | Task: 0.0360 | Monitor: 0.9367 | LR: 1.00e-02 | No improvement: 9/1000
Epoch   150 | Loss: 0.0357 | Task: 0.0356 | Monitor: 0.9352 | LR: 1.00e-02 | No improvement: 21/1000
Epoch   200 | Loss: 0.0355 | Task: 0.0354 | Monitor: 0.9334 | LR: 1.00e-02 | No improvement: 2/1000
Epoch   250 | Loss: 0.0351 | Task: 0.0350 | Monitor: 0.9305 | LR: 1.00e-02 | No improvement: 8/1000
Epoch   300 | Loss: 0.0348 | Task: 0.0346 | Monitor: 0.9286 | LR: 1.00e-02 | No improvement: 9/1000
Epoch   350 | Loss: 0.0345 | Task: 0.0344 | Monitor: 0.9284 | LR: 5.00e-03 | No improvement: 5/1000
Epoch   400 | Loss: 0.0344 | Task: 0.0342 | Monitor: 0.9271 | LR: 1.25e-03 | No improvement: 55/1000
Epoch   450 | Loss: 0.0343 | Task: 0.0342 | Monitor: 0.9281 | LR: 3.13e-04 | No improvement: 105/1

In [ ]:
# After a forward pass, check if users are actually using different layers
u0 = rating_model.user_embedding.weight.detach()
layer_w = F.softmax(rating_model.user_layer_attn(u0), dim=1).detach().cpu()  # [N, L+1]

print("Mean weight per layer:", layer_w.mean(dim=0))   # should not be uniform
print("Std weight per layer: ", layer_w.std(dim=0))    # low std = no differentiation

# Find users who lean heavily on early vs late layers
early_users = layer_w[:, 0].topk(5).indices   # users who weight layer 0 most
late_users  = layer_w[:, -1].topk(5).indices  # users who weight final layer most
print("Early-layer users:", early_users)
print("Late-layer users: ", late_users)